In [6]:
import pandas as pd
import numpy as np
import glob
from pathlib import Path

In [7]:
CORRECCION_NOMBRES_GEE = {
    "Brice¥o": "Briceño",
    "Pisva": "Pisba",
    "Santamaria": "Santa Maria",
}


def fix_municipio_names(df: pd.DataFrame, muni_col: str = "municipio") -> pd.DataFrame:
    """Corrige nombres de municipio conocidos con error en las fuentes GEE."""
    df = df.copy()
    df[muni_col] = df[muni_col].replace(CORRECCION_NOMBRES_GEE)
    return df


def filter_complete_years(df: pd.DataFrame, date_col: str = "fecha", muni_col: str = "municipio",
                           freq: str = "daily", min_coverage: float = 0.9) -> pd.DataFrame:
    """
    Excluye años incompletos (típicamente el año en curso, con datos parciales) para que:
      (a) el acumulado/promedio anual no salga subestimado, y
      (b) no se contamine la media histórica usada en historical_anomaly(), que de lo
          contrario mezclaría un año parcial como si fuera un año completo.

    freq: "daily" (CHIRPS/ERA5, ~365 obs/año), "modis16" (MODIS, 23 obs/año),
          "monthly" (TerraClimate, 12 obs/año).
    """
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df["_anio_tmp"] = df[date_col].dt.year

    expected = {"daily": 365, "modis16": 23, "monthly": 12}[freq]
    conteo = df.groupby(["_anio_tmp", muni_col]).size().groupby("_anio_tmp").min()
    anios_incompletos = conteo[conteo < expected * min_coverage].index.tolist()

    if anios_incompletos:
        print(f"[AVISO] Año(s) excluido(s) por estar incompletos (probablemente en curso): "
              f"{anios_incompletos}")
        df = df[~df["_anio_tmp"].isin(anios_incompletos)]

    return df.drop(columns=["_anio_tmp"])


In [8]:
RAW_DIR = Path("../data_raw")
CLEAN_DIR = Path("../data_clean")
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)

# Preprocesamiento

## Terraclimate - Escalado, Duplicados, Nulos

In [9]:
SCALE_FACTORS = {
    "tmmx": 0.1, "tmmn": 0.1,
    "aet": 0.1, "pet": 0.1, "def": 0.1, "soil": 0.1, "ro": 0.1,
    "srad": 0.1,
    "vap": 0.001,
    "vpd": 0.01, "pdsi": 0.01,
    # "pr" se deja sin escalar (ya viene en mm)
}

PLAUSIBLE_RANGES = {
    "tmmx": (-5, 45), "tmmn": (-15, 30), "pr": (0, 1000),
    "aet": (0, 300), "pet": (0, 300), "def": (0, 300),
    "soil": (0, 500), "ro": (0, 500), "srad": (0, 400),
    "vap": (0, 5), "vpd": (0, 5), "pdsi": (-10, 10),
}

tc_raw = pd.read_csv(RAW_DIR / "TerraClimate_Boyaca_57_Municipios_2007_2024.csv")
tc = fix_municipio_names(tc_raw)

for col, factor in SCALE_FACTORS.items():
    if col in tc.columns:
        tc[col] = tc[col] * factor

n_dup = tc.duplicated(subset=["fecha", "municipio"]).sum()
if n_dup > 0:
    print(f"[AVISO] {n_dup} duplicados eliminados en TerraClimate")
    tc = tc.drop_duplicates(subset=["fecha", "municipio"], keep="first")

for col, (lo, hi) in PLAUSIBLE_RANGES.items():
    if col in tc.columns:
        n_out = ((tc[col] < lo) | (tc[col] > hi)).sum()
        if n_out > 0:
            print(f"[REVISAR] {col}: {n_out} valores fuera de [{lo}, {hi}] (no se eliminan, solo se marcan)")

print(f"\nTerraClimate limpio: {tc.shape[0]} filas, {tc['municipio'].nunique()} municipios")
tc.describe().T[["min", "mean", "max"]]

[REVISAR] pdsi: 77 valores fuera de [-10, 10] (no se eliminan, solo se marcan)

TerraClimate limpio: 12312 filas, 57 municipios


,min,mean,max
tmmx,12.358954,22.985648,36.212795
tmmn,3.562388,13.798132,27.341919
pr,0.750683,169.495322,895.280061
aet,11.230510,86.455798,135.337451
pet,58.614007,93.518662,163.859506
def,0.000000,7.061822,149.762630
soil,7.289675,138.927093,241.911359
ro,0.000000,8.307483,81.472273
srad,120.250527,186.562764,284.016944
vap,0.848132,1.732824,3.097726


In [10]:
tc.to_csv(CLEAN_DIR / "TerraClimate_clean.csv", index=False)
print("Guardado: TerraClimate_clean.csv")

Guardado: TerraClimate_clean.csv


## CHIRPS - consolidacion de docs

In [11]:
chirps_files = sorted(glob.glob(str(RAW_DIR / "CHIRPS_Boyaca_*.csv")))
print(f"Archivos CHIRPS encontrados: {len(chirps_files)}")
for f in chirps_files:
    print(" -", Path(f).name)

chirps = pd.concat([pd.read_csv(f) for f in chirps_files], ignore_index=True)
chirps = fix_municipio_names(chirps)
chirps = filter_complete_years(chirps, freq="daily")
chirps["fecha"] = pd.to_datetime(chirps["fecha"])

n_dup = chirps.duplicated(subset=["fecha", "municipio"]).sum()
if n_dup > 0:
    print(f"[AVISO] {n_dup} duplicados eliminados en CHIRPS")
    chirps = chirps.drop_duplicates(subset=["fecha", "municipio"], keep="first")

print(f"\nCHIRPS limpio: {chirps.shape[0]} filas | {chirps['municipio'].nunique()} municipios "
      f"| {chirps['fecha'].min().date()} a {chirps['fecha'].max().date()}")
chirps["precipitation_chirps"].describe()

Archivos CHIRPS encontrados: 4
 - CHIRPS_Boyaca_Diario_2007_2011.csv
 - CHIRPS_Boyaca_Diario_2012_2016.csv
 - CHIRPS_Boyaca_Diario_2017_2021.csv
 - CHIRPS_Boyaca_Diario_2022_Ultimo_Dato.csv
[AVISO] Año(s) excluido(s) por estar incompletos (probablemente en curso): [2026]

CHIRPS limpio: 395580 filas | 57 municipios | 2007-01-01 a 2025-12-31


count    395580.000000
mean          5.302134
std           9.545777
min           0.000000
25%           0.000000
50%           0.000000
75%           7.386264
max         184.618758
Name: precipitation_chirps, dtype: float64

In [12]:
chirps.to_csv(CLEAN_DIR / "CHIRPS_clean.csv", index=False)
print("Guardado: CHIRPS_clean.csv")

Guardado: CHIRPS_clean.csv


## ERA5 - Consolidacion

In [13]:
era5_files = sorted(glob.glob(str(RAW_DIR / "ERA5_Boyaca_*.csv")))
print(f"Archivos ERA5 encontrados: {len(era5_files)}")

era5 = pd.concat([pd.read_csv(f) for f in era5_files], ignore_index=True)
era5 = fix_municipio_names(era5)
era5 = filter_complete_years(era5, freq="daily")
era5["fecha"] = pd.to_datetime(era5["fecha"])

n_dup = era5.duplicated(subset=["fecha", "municipio"]).sum()
if n_dup > 0:
    print(f"[AVISO] {n_dup} duplicados eliminados en ERA5")
    era5 = era5.drop_duplicates(subset=["fecha", "municipio"], keep="first")

print(f"\nERA5 limpio: {era5.shape[0]} filas | {era5['municipio'].nunique()} municipios "
      f"| {era5['fecha'].min().date()} a {era5['fecha'].max().date()}")
era5.describe().T[["min", "mean", "max"]]

Archivos ERA5 encontrados: 20
[AVISO] Año(s) excluido(s) por estar incompletos (probablemente en curso): [2026]

ERA5 limpio: 395580 filas | 57 municipios | 2007-01-01 a 2025-12-31


,min,mean,max
fecha,2007-01-01 00:00:00,2016-07-01 12:00:00.000000512,2025-12-31 00:00:00
temp_mean,8.717952,16.531989,32.236589
temp_min,-0.215898,12.917248,27.63574
temp_max,11.075774,20.787318,39.000544
dewpoint,-4.184788,13.353056,24.374715
relative_humidity,33.367359,81.977958,99.460045
precipitation,-0.000012,6.083153,180.498728
soil_moisture,0.21845,0.427523,0.508541
solar_radiation,2.350423,17.69047,29.388846
evaporation,-5.902401,-3.103333,-0.617129


In [14]:
era5.to_csv(CLEAN_DIR / "ERA5_clean.csv", index=False)
print("Guardado: ERA5_clean.csv")

Guardado: ERA5_clean.csv


## MODIS MOD13Q1 - Consolidacion

In [15]:
modis_files = sorted(glob.glob(str(RAW_DIR / "MOD13Q1_Boyaca_*.csv")))
print(f"Archivos MODIS encontrados: {len(modis_files)}")

modis = pd.concat([pd.read_csv(f) for f in modis_files], ignore_index=True)
modis = fix_municipio_names(modis)
modis = filter_complete_years(modis, freq="modis16", min_coverage=0.85)
modis["fecha"] = pd.to_datetime(modis["fecha"])

n_dup = modis.duplicated(subset=["fecha", "municipio"]).sum()
if n_dup > 0:
    print(f"[AVISO] {n_dup} duplicados eliminados en MODIS")
    modis = modis.drop_duplicates(subset=["fecha", "municipio"], keep="first")

# Confirmar periodicidad esperada (~23 observaciones/año por compuesto de 16 días)
obs_por_anio = modis[modis["municipio"] == "Almeida"].groupby(modis["fecha"].dt.year).size()
print("\nObservaciones por año (municipio de referencia: Almeida):")
print(obs_por_anio)

print(f"\nMODIS limpio: {modis.shape[0]} filas | {modis['municipio'].nunique()} municipios "
      f"| {modis['fecha'].min().date()} a {modis['fecha'].max().date()}")
modis[["NDVI", "EVI", "SummaryQA"]].describe()

Archivos MODIS encontrados: 2
[AVISO] Año(s) excluido(s) por estar incompletos (probablemente en curso): [2026]

Observaciones por año (municipio de referencia: Almeida):
fecha
2007    23
2008    23
2009    23
2010    23
2011    23
2012    23
2013    23
2014    23
2015    23
2016    23
2017    23
2018    23
2019    23
2020    23
2021    23
2022    23
2023    23
2024    23
2025    23
dtype: int64

MODIS limpio: 24909 filas | 57 municipios | 2007-01-01 a 2025-12-19


,NDVI,EVI,SummaryQA
count,24909.000000,24909.000000,24909.000000
mean,0.611255,0.389427,1.422672
std,0.164621,0.102627,0.969640
min,0.033457,0.028011,0.000000
25%,0.529283,0.330562,0.538830
50%,0.658597,0.405476,1.410039
75%,0.735385,0.463359,2.257310
max,0.873665,0.695234,3.000000


In [16]:
modis.to_csv(CLEAN_DIR / "MODIS_clean.csv", index=False)
print("Guardado: MODIS_clean.csv")

Guardado: MODIS_clean.csv


## STRM

In [17]:
srtm = pd.read_csv(RAW_DIR / "SRTM_Boyaca_57_Municipios.csv")
srtm = fix_municipio_names(srtm)

print(f"SRTM: {srtm.shape[0]} municipios")
srtm.describe().T[["min", "mean", "max"]]

SRTM: 57 municipios


,min,mean,max
elevacion_media_m,218.134531,1903.566887,3226.903122
elevacion_min_m,107.000000,1084.368421,2132.000000
elevacion_max_m,1506.000000,2912.631579,4764.000000
elevacion_std_m,138.547813,376.706762,972.418844


In [18]:
srtm.to_csv(CLEAN_DIR / "SRTM_clean.csv", index=False)
print("Guardado: SRTM_clean.csv")

Guardado: SRTM_clean.csv


## Datos de produccion - Agronet

In [19]:
def parse_numero_es(serie: pd.Series) -> pd.Series:
    """Convierte '1.374,50' (formato es-CO) a 1374.50 (float)."""
    return (
        serie.astype(str)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .astype(float)
    )

agronet = pd.read_csv(RAW_DIR / "detalle_agricola_municipal_municipio_20260816.csv", encoding="utf-8-sig")
agronet["Municipio"] = agronet["Municipio"].str.strip().str.title()

for col in ["Área Cosechada", "Área Sembrada", "Producción", "Rendimiento"]:
    agronet[col] = parse_numero_es(agronet[col])

agronet = agronet.rename(columns={
    "Año": "anio", "Municipio": "municipio",
    "Área Cosechada": "area_cosechada_ha", "Área Sembrada": "area_sembrada_ha",
    "Producción": "produccion_ton", "Rendimiento": "rendimiento_t_ha",
})[["municipio", "anio", "area_cosechada_ha", "area_sembrada_ha", "produccion_ton", "rendimiento_t_ha"]]

# Chequeo de consistencia interna
agronet["rendimiento_calculado"] = agronet["produccion_ton"] / agronet["area_cosechada_ha"]
agronet["diff_rendimiento_pct"] = (
    (agronet["rendimiento_calculado"] - agronet["rendimiento_t_ha"]).abs()
    / agronet["rendimiento_t_ha"] * 100
)
inconsistentes = agronet[agronet["diff_rendimiento_pct"] > 10]
print(f"[REVISAR] {len(inconsistentes)} filas con posible error de captura (diff > 10%)")

print(f"\nAgronet: {agronet.shape[0]} filas | {agronet['municipio'].nunique()} municipios "
      f"| años {sorted(agronet['anio'].unique())}")
print(f"Completitud del panel: {len(agronet) / (agronet['municipio'].nunique() * agronet['anio'].nunique()):.1%}")
agronet.head()

[REVISAR] 0 filas con posible error de captura (diff > 10%)

Agronet: 827 filas | 57 municipios | años [np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Completitud del panel: 76.4%


,municipio,anio,area_cosechada_ha,area_sembrada_ha,produccion_ton,rendimiento_t_ha,rendimiento_calculado,diff_rendimiento_pct
0,Almeida,2014,65.43,72.48,45.29,0.69,0.692190,0.317410
1,Almeida,2022,70.47,72.67,81.11,1.15,1.150986,0.085760
2,Almeida,2019,68.00,71.00,83.00,1.22,1.220588,0.048216
3,Almeida,2025,40.83,41.83,49.07,1.20,1.201812,0.151033
4,Almeida,2012,2.00,32.00,1.20,0.60,0.600000,0.000000


In [20]:
agronet.to_csv(CLEAN_DIR / "Agronet_target_clean.csv", index=False)
print("Guardado: Agronet_target_clean.csv")

Guardado: Agronet_target_clean.csv


# Verificaciones adicionales

In [21]:
fuentes = {
    "TerraClimate": tc, "CHIRPS": chirps, "ERA5": era5,
    "MODIS": modis, "SRTM": srtm, "Agronet": agronet,
}

base = set(srtm["municipio"])
for nombre, df in fuentes.items():
    s = set(df["municipio"])
    faltantes = base - s
    sobrantes = s - base
    estado = "OK" if not faltantes and not sobrantes else "REVISAR"
    print(f"[{estado}] {nombre}: {len(s)} municipios | faltan: {faltantes or '-'} | extra: {sobrantes or '-'}")

[OK] TerraClimate: 57 municipios | faltan: - | extra: -
[OK] CHIRPS: 57 municipios | faltan: - | extra: -
[OK] ERA5: 57 municipios | faltan: - | extra: -
[OK] MODIS: 57 municipios | faltan: - | extra: -
[OK] SRTM: 57 municipios | faltan: - | extra: -
[OK] Agronet: 57 municipios | faltan: - | extra: -


In [22]:
# Revision de nulos
archivos_clean = {
    "TerraClimate": ("TerraClimate_clean.csv", "fecha"),
    "CHIRPS": ("CHIRPS_clean.csv", "fecha"),
    "ERA5": ("ERA5_clean.csv", "fecha"),
    "MODIS": ("MODIS_clean.csv", "fecha"),
    "SRTM": ("SRTM_clean.csv", None),
    "Agronet": ("Agronet_target_clean.csv", "anio"),
}

resumen_qc = []
for nombre, (archivo, col_fecha) in archivos_clean.items():
    df = pd.read_csv(CLEAN_DIR / archivo)
    n_dup_exactos = df.duplicated().sum()
    n_nulos = df.isnull().sum().sum()
    cols_con_nulos = df.columns[df.isnull().any()].tolist()

    fila = {
        "fuente": nombre,
        "filas": df.shape[0],
        "columnas": df.shape[1],
        "municipios": df["municipio"].nunique() if "municipio" in df.columns else "-",
        "duplicados_exactos": n_dup_exactos,
        "total_nulos": n_nulos,
        "columnas_con_nulos": ", ".join(cols_con_nulos) if cols_con_nulos else "ninguna",
    }
    if col_fecha == "fecha":
        df[col_fecha] = pd.to_datetime(df[col_fecha])
        fila["rango"] = f"{df[col_fecha].min().date()} a {df[col_fecha].max().date()}"
    elif col_fecha == "anio":
        fila["rango"] = f"{df[col_fecha].min()} a {df[col_fecha].max()}"
    else:
        fila["rango"] = "estático (sin fecha)"

    resumen_qc.append(fila)

qc_df = pd.DataFrame(resumen_qc)
qc_df

,fuente,filas,columnas,municipios,duplicados_exactos,total_nulos,columnas_con_nulos,rango
0,TerraClimate,12312,14,57,0,0,ninguna,2007-01-01 a 2024-12-01
1,CHIRPS,395580,3,57,0,0,ninguna,2007-01-01 a 2025-12-31
2,ERA5,395580,11,57,0,0,ninguna,2007-01-01 a 2025-12-31
3,MODIS,24909,5,57,0,0,ninguna,2007-01-01 a 2025-12-19
4,SRTM,57,5,57,0,0,ninguna,estático (sin fecha)
5,Agronet,827,8,57,0,44,"rendimiento_calculado, diff_rendimiento_pct",2007 a 2025


In [23]:
# Duplicados por clave lógica (municipio + fecha/año), no solo duplicados de fila exacta
print("Duplicados por clave lógica (municipio + fecha/año):")
for nombre, (archivo, col_fecha) in archivos_clean.items():
    if col_fecha is None:
        continue
    df = pd.read_csv(CLEAN_DIR / archivo)
    clave = ["municipio", col_fecha]
    n_dup = df.duplicated(subset=clave).sum()
    estado = "OK" if n_dup == 0 else "REVISAR"
    print(f"  [{estado}] {nombre}: {n_dup} duplicados en {clave}")

Duplicados por clave lógica (municipio + fecha/año):
  [OK] TerraClimate: 0 duplicados en ['municipio', 'fecha']
  [OK] CHIRPS: 0 duplicados en ['municipio', 'fecha']
  [OK] ERA5: 0 duplicados en ['municipio', 'fecha']
  [OK] MODIS: 0 duplicados en ['municipio', 'fecha']
  [OK] Agronet: 0 duplicados en ['municipio', 'anio']
